# Notebook 01 — Exploratory Data Analysis
## Canadian Wildfire Spread Prediction — BC Dataset

**Purpose**: Understand the assembled dataset before modelling.

Sections:
1. Dataset summary (patch counts, split sizes, years)
2. Class distribution (burning vs. non-burning pixels)
3. Channel statistics and correlations
4. FWI system feature distributions
5. Geographic coverage map (BC fire locations)
6. Year-by-year fire season breakdown
7. Example patches (visual inspection)

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config import CHANNELS, N_CHANNELS, PATCH_SIZE
from src.utils.visualization import bc_fire_coverage_map

plt.style.use('seaborn-v0_8-whitegrid')
PATCHES_DIR = Path('../data/processed/patches')
SPLITS_DIR  = Path('../data/processed/splits')
STATS_DIR   = Path('../data/processed/stats')

print(f'N_CHANNELS = {N_CHANNELS}')
print(f'PATCH_SIZE = {PATCH_SIZE}')

## 1. Dataset Summary

In [ ]:
splits = {}
all_records = []
for split in ['train', 'val', 'test']:
    p = SPLITS_DIR / f'{split}.json'
    if p.exists():
        with open(p) as f:
            recs = json.load(f)
        splits[split] = recs
        for r in recs:
            r['split'] = split
        all_records.extend(recs)

df = pd.DataFrame(all_records)
print(f"Total patches: {len(df):,}")
print(f"\nBy split:")
print(df.groupby('split')['patch_id'].count().to_string())
print(f"\nBy year:")
print(df.groupby('year')['patch_id'].count().to_string())

## 2. Class Distribution (Burning vs Non-Burning Pixels)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Burn fraction histogram
ax = axes[0]
for split_name, color in [('train','steelblue'), ('val','orange'), ('test','red')]:
    if split_name in splits:
        fracs = [r['burn_fraction'] for r in splits[split_name]]
        ax.hist(fracs, bins=40, alpha=0.6, label=split_name, color=color, density=True)
ax.set_xlabel('Burn fraction per patch')
ax.set_ylabel('Density')
ax.set_title('Distribution of Burn Fraction per Patch')
ax.legend()

# Overall positive class rate by split
ax2 = axes[1]
split_pos_rates = {}
for split_name in ['train', 'val', 'test']:
    if split_name in splits:
        fracs = [r['burn_fraction'] for r in splits[split_name]]
        split_pos_rates[split_name] = np.mean(fracs)
ax2.bar(split_pos_rates.keys(), split_pos_rates.values(), color=['steelblue','orange','red'])
ax2.set_ylabel('Mean burn fraction')
ax2.set_title('Mean Positive Pixel Rate by Split')
ax2.set_ylim([0, 0.5])
for k, v in split_pos_rates.items():
    ax2.text(list(split_pos_rates.keys()).index(k), v + 0.005, f'{v:.1%}', ha='center')

plt.tight_layout()
plt.show()
print(f"Note: ~{split_pos_rates.get('train', 0):.1%} positive pixels in training split")
print(f"Compare NDWS: ~2-5% positive pixels (patch-centred strategy improves signal)")

## 3. Channel Statistics

In [ ]:
stats_path = STATS_DIR / 'channel_stats.json'
if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)
    
    stats_df = pd.DataFrame([
        {'channel': ch, 'mean': v['mean'], 'std': v['std']}
        for ch, v in stats.items()
    ])
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    ax = axes[0]
    ax.barh(stats_df['channel'], stats_df['mean'])
    ax.set_xlabel('Mean value (raw, un-normalised)')
    ax.set_title('Per-Channel Mean (Training Split)')
    
    ax2 = axes[1]
    ax2.barh(stats_df['channel'], stats_df['std'], color='orange')
    ax2.set_xlabel('Standard deviation')
    ax2.set_title('Per-Channel Std (Training Split)')
    
    plt.tight_layout()
    plt.show()
else:
    print('Channel stats not found. Run: python scripts/assemble_patches.py')

## 4. FWI System Feature Distributions

The FWI system (Fine Fuel Moisture Code, Duff Moisture Code, Drought Code, 
Initial Spread Index, Build-up Index, Fire Weather Index) is the operational
standard used by NRCan and BC Wildfire Service.

Source: ERA5-HRS global FWI reanalysis (McElhinny et al. 2020, doi:10.5281/zenodo.3626193)
Using overwintered DC values — recommended for western Canada.

In [ ]:
# Sample patches to get FWI distributions
fwi_channels = ['FFMC', 'DMC', 'DC', 'ISI', 'BUI', 'FWI']
fwi_idx = {ch: CHANNELS.index(ch) for ch in fwi_channels if ch in CHANNELS}

sample_recs = list(splits.get('train', []))[:200]  # sample 200 patches
fwi_data = {ch: [] for ch in fwi_channels}

for rec in sample_recs:
    p = PATCHES_DIR / f"{rec['patch_id']}.npz"
    if not p.exists():
        continue
    data = np.load(p)
    X = data['X']  # (C, H, W)
    for ch in fwi_channels:
        idx = fwi_idx.get(ch)
        if idx is not None and idx < X.shape[0]:
            fwi_data[ch].extend(X[idx].ravel().tolist())

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, ch in enumerate(fwi_channels):
    ax = axes[i // 3][i % 3]
    if fwi_data[ch]:
        ax.hist(fwi_data[ch], bins=50, color='firebrick', alpha=0.7)
    ax.set_title(ch)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

plt.suptitle('FWI System Channel Distributions (Training Split Sample)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Geographic Coverage Map

Interactive folium map of patch locations coloured by fire year.
Note the concentration in BC's Interior and Northern regions — fire-prone forest zones.

In [ ]:
try:
    coverage_map = bc_fire_coverage_map(
        all_records,
        save_path=Path('../data/processed/bc_coverage_map.html')
    )
    coverage_map  # display inline in Jupyter
except Exception as e:
    print(f'Map display error: {e}')
    print('Install folium: pip install folium')

## 6. Year-by-Year Fire Season Breakdown

In [ ]:
if len(df) > 0:
    year_summary = df.groupby('year').agg(
        n_patches=('patch_id', 'count'),
        mean_burn_frac=('burn_fraction', 'mean'),
        n_unique_fires=('fire_id', 'nunique'),
    ).reset_index()
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    ax = axes[0]
    colors = ['red' if y == 2023 else ('orange' if y in [2021, 2022] else 'steelblue')
              for y in year_summary['year']]
    ax.bar(year_summary['year'], year_summary['n_patches'], color=colors)
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of patches')
    ax.set_title('Patches per Year')
    ax.tick_params(axis='x', rotation=45)
    # Legend
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color='steelblue', label='Train (2012–2020)'),
        Patch(color='orange', label='Val (2021–2022)'),
        Patch(color='red', label='Test (2023)'),
    ], fontsize=8)
    
    ax2 = axes[1]
    ax2.bar(year_summary['year'], year_summary['n_unique_fires'], color=colors)
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Number of unique fires')
    ax2.set_title('Unique Fire Events per Year')
    ax2.tick_params(axis='x', rotation=45)
    
    ax3 = axes[2]
    ax3.bar(year_summary['year'], year_summary['mean_burn_frac'] * 100, color=colors)
    ax3.set_xlabel('Year')
    ax3.set_ylabel('Mean burn fraction (%)')
    ax3.set_title('Fire Intensity by Year (Mean Burn %)')
    ax3.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print('\nKey fire seasons:')
    print('  2017: BC record at the time — 1.2M ha burned')
    print('  2021: Lytton Creek (largest BC fire up to that point)')
    print('  2023: Donnie Creek — largest BC wildfire on record (~2.84M ha) [TEST SPLIT]')

## 7. Example Patch Visualisation

Visual inspection: input channels and target burn mask for a sample patch.

In [ ]:
sample_recs = list(splits.get('test', splits.get('train', [])))[:1]
for rec in sample_recs:
    p = PATCHES_DIR / f"{rec['patch_id']}.npz"
    if not p.exists():
        print(f'Patch not found: {p}')
        continue
    data = np.load(p)
    X = data['X']   # (C, H, W)
    y = data['y']   # (H, W)
    
    # Show key channels + target
    channels_to_show = ['elevation', 'FWI', 'NDVI', 'PrevFireMask']
    ch_idx = {ch: CHANNELS.index(ch) for ch in channels_to_show if ch in CHANNELS}
    
    n_cols = len(ch_idx) + 1  # +1 for target
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    
    for i, (ch_name, idx) in enumerate(ch_idx.items()):
        ax = axes[i]
        cmap = 'RdYlGn_r' if ch_name == 'FWI' else ('Greens' if ch_name == 'NDVI' else 'terrain')
        im = ax.imshow(X[idx], cmap=cmap, origin='upper')
        ax.set_title(ch_name)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    # Target
    axes[-1].imshow(y, cmap='hot', vmin=0, vmax=1, origin='upper')
    axes[-1].set_title(f'Target (next-day burn)\n{rec["date"]}')
    axes[-1].axis('off')
    
    plt.suptitle(
        f"Fire: {rec['fire_id']} | Date: {rec['date']} | "
        f"Burn fraction: {rec['burn_fraction']:.1%}",
        fontsize=11
    )
    plt.tight_layout()
    plt.show()

## 8. Channel Correlation Matrix

Identifies redundant channels and potential multicollinearity issues for the RF baseline.

In [ ]:
sample_patches = list(splits.get('train', []))[:50]
channel_samples = []

for rec in sample_patches:
    p = PATCHES_DIR / f"{rec['patch_id']}.npz"
    if not p.exists():
        continue
    X = np.load(p)['X']  # (C, H, W)
    # Sample 100 random pixels per patch
    flat = X.reshape(X.shape[0], -1).T  # (H*W, C)
    idx = np.random.choice(len(flat), size=min(100, len(flat)), replace=False)
    channel_samples.append(flat[idx])

if channel_samples:
    all_pixels = np.concatenate(channel_samples, axis=0)
    corr_df = pd.DataFrame(all_pixels, columns=CHANNELS).corr()
    
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        corr_df, annot=False, fmt='.2f', cmap='coolwarm',
        center=0, vmin=-1, vmax=1, ax=ax, square=True,
        cbar_kws={'label': 'Pearson correlation'}
    )
    ax.set_title('Channel Correlation Matrix (Training Pixel Sample)', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    # Note high correlations
    high_corr = []
    for i in range(len(CHANNELS)):
        for j in range(i+1, len(CHANNELS)):
            c = corr_df.iloc[i, j]
            if abs(c) > 0.7:
                high_corr.append((CHANNELS[i], CHANNELS[j], round(c, 3)))
    if high_corr:
        print('High correlations (|r| > 0.7):')
        for a, b, c in sorted(high_corr, key=lambda x: -abs(x[2])):
            print(f'  {a} ↔ {b}: {c}')
else:
    print('No patches available. Run scripts/assemble_patches.py first.')

---

## Summary

Key observations to carry into modelling:

1. **Class imbalance**: ~10–20% burning pixels at patch level (better than NDWS ~2–5% due to patch-centred sampling)
2. **FWI distributions**: Note FWI values during extreme seasons (2017, 2021, 2023) — look for out-of-distribution samples in test split
3. **FWI source**: ERA5-HRS overwintered DC (2012–2018, Zenodo) + CWFIS operational (2019–2023)
4. **2023 test split**: Includes Donnie Creek — the largest BC wildfire on record. This is a genuinely hard generalization test.
5. **BurnP3+ channel**: Currently zero-filled (no BurnP3+ runs). Ablation in Phase 2 will measure its value as a physics prior.

Next: `02_baseline_rf.ipynb`